# Portfolio optimization

In the [Measure Impact](../../measure-impact/index.md) stage we produced causal estimates. In the [Evaluate Evidence](../../evaluate-evidence/index.md) stage we assessed how much to trust them. This lecture addresses the final question: given confidence-weighted return estimates for a set of initiatives, which ones should we fund?

The answer requires decision theory. An organization faces a binary selection problem — fund or skip each initiative — subject to a budget constraint, with returns that depend on an unknown state of nature. The confidence scores from the **EVALUATE** stage enter directly: lower confidence penalizes an initiative's projected returns, pulling them toward the worst case. This creates a built-in incentive for better measurement — investing in evidence quality raises an initiative's effective returns and improves its chances of selection.

Part I develops the mathematical framework following Eisenhauer (2025). Part II applies it end-to-end using the [**impact-engine-allocate**](https://eisenhauerio.github.io/tools-impact-engine-allocate/) package on mock portfolio data matching the paper's simulation example.

---

## Part I: Theory

### 1. The decision problem

An organization has $N$ investment initiatives indexed by the set $I = \{1, \ldots, N\}$. Each initiative $i$ has a known cost $b_i$ and an uncertain return that depends on which scenario materializes. The total budget is $B$.

The decision is binary: for each initiative $i$, the decision variable $x_i \in \{0, 1\}$ indicates whether to fund it. The portfolio vector $\mathbf{x} = (x_1, \ldots, x_N)$ collects all selection decisions. The constraints are:

$$x_i \in \{0, 1\} \quad \forall i \in I \tag{2}$$

$$\sum_{i=1}^{N} b_i x_i \leq B \tag{5}$$

Equation (2) makes the problem a binary integer program — each initiative is either fully funded or not. Equation (5) ensures total spending does not exceed the budget. The objective function depends on the decision rule, which we develop in §4 and §5.

### 2. Scenario-dependent returns

Returns are uncertain because they depend on which state of nature materializes. The set of scenarios $S = \{s_1, \ldots, s_M\}$ represents the possible states. In the standard configuration, $S = \{s_{\text{best}}, s_{\text{med}}, s_{\text{worst}}\}$ — three scenarios spanning the range of plausible outcomes.

For each initiative $i$ and scenario $s_j$, the **baseline return** $R_{ij}$ is the net return (revenue minus cost) if $i$ is funded and scenario $s_j$ materializes. The **worst-case return** is:

$$R_i^{\min} = \min_{s_j \in S} R_{ij}$$

These baseline returns come from the **MEASURE** stage — they are the causal effect estimates produced by the Impact Engine. The key question is how much to trust them, which is where the **EVALUATE** stage enters.

### 3. The confidence penalty

The confidence score $c_i \in [0, 1]$ from the **EVALUATE** stage quantifies how much to trust initiative $i$'s return estimates. A low confidence score means the evidence is weak — the true returns could be far from the estimates. The framework translates this epistemic uncertainty into a penalty on projected returns.

The **penalty factor** $\gamma_i$ is a monotonically decreasing function of confidence:

$$\gamma_i = f(c_i) = 1 - c_i$$

The **effective return** $\tilde{R}_{ij}$ blends each scenario return toward the worst case, weighted by the penalty:

$$\tilde{R}_{ij} = (1 - \gamma_i) R_{ij} + \gamma_i R_i^{\min} \quad \forall s_j \in S$$

When confidence is perfect ($c_i = 1$, $\gamma_i = 0$), the effective return equals the baseline: $\tilde{R}_{ij} = R_{ij}$. When confidence is zero ($c_i = 0$, $\gamma_i = 1$), all scenarios collapse to the worst case: $\tilde{R}_{ij} = R_i^{\min}$. This creates a direct incentive for better measurement — improving evidence quality raises an initiative's effective returns.

A **minimum confidence threshold** $c_{\min}$ excludes initiatives with insufficient evidence:

$$c_i \geq c_{\min} \cdot x_i \quad \forall i \in I \tag{3}$$

Any initiative with $c_i < c_{\min}$ is ineligible for selection regardless of its projected returns.

### 4. Minimax regret optimization

The minimax regret rule selects the portfolio that minimizes the worst-case disappointment across all scenarios. To define regret, we first need the **optimal benchmark** — the best achievable effective return under each scenario:

$$V_j^* = \max_{\mathbf{x}} \sum_{i=1}^{N} \tilde{R}_{ij} x_i \quad \text{s.t. budget and binary constraints}$$

The **regret** of portfolio $\mathbf{x}$ under scenario $s_j$ is the gap between what was achievable and what the portfolio delivers:

$$\text{Regret}_j(\mathbf{x}) = V_j^* - \sum_{i=1}^{N} \tilde{R}_{ij} x_i$$

The minimax regret formulation introduces an auxiliary variable $\theta$ that bounds the maximum regret:

$$\min_{\mathbf{x}, \theta} \quad \theta \tag{1}$$

$$\theta \geq V_j^* - \sum_{i=1}^{N} \tilde{R}_{ij} x_i \quad \forall s_j \in S \tag{4}$$

subject to the binary constraint (2), the confidence threshold (3), and the budget constraint (5). An optional **downside safeguard** guarantees a minimum portfolio return under the worst case:

$$\sum_{i=1}^{N} R_i^{\min} x_i \geq R_{\min}^{\text{portfolio}} \tag{6}$$

The minimax regret rule is conservative — it protects against the scenario where the chosen portfolio performs worst relative to what was possible. The optimal $\theta^*$ tells the decision-maker the maximum regret they face.

### 5. Bayesian decision rule

An alternative to minimax regret is the Bayesian decision rule, which assigns probability weights $w_j$ to each scenario and maximizes the weighted expected return:

$$\max_{\mathbf{x}} \sum_{j=1}^{M} w_j \sum_{i=1}^{N} \tilde{R}_{ij} x_i$$

subject to the same constraints (2), (3), (5), and (6). The weights satisfy $w_j \geq 0$ and $\sum_j w_j = 1$.

Different weight profiles express different beliefs about which scenario is most likely:

| Profile | $w_{\text{best}}$ | $w_{\text{med}}$ | $w_{\text{worst}}$ | Interpretation |
|---------|:-:|:-:|:-:|----------------|
| Optimistic | 0.50 | 0.30 | 0.20 | Upside scenarios dominate |
| Balanced | 0.33 | 0.34 | 0.33 | Equal uncertainty across scenarios |
| Pessimistic | 0.20 | 0.30 | 0.50 | Worst case most likely |

The Bayesian rule is less conservative than minimax regret — it allows the decision-maker to express beliefs about scenario likelihoods rather than optimizing for the worst case. Comparing the two rules on the same portfolio reveals how much the portfolio choice depends on the decision-maker's attitude toward uncertainty.

---

## Part II: Application

In [ ]:
# Standard Library
import copy
import inspect

# Third-party
import pandas as pd
import yaml
from impact_engine_allocate import BayesianSolver, solve_minimax_regret
from impact_engine_allocate.solver import (
    calculate_effective_returns,
    calculate_gamma,
    preprocess,
)
from IPython.display import Code

# Local
from support import (
    create_mock_portfolio,
    display_solver_result,
    plot_confidence_penalty,
    plot_effective_returns_heatmap,
    plot_portfolio_comparison,
    plot_scenario_returns,
)

## 1. Initiative data

We construct a mock portfolio of five initiatives matching the simulation example in Eisenhauer (2025), Table 1. Each initiative has a cost, three scenario returns (best, median, worst), and a confidence score from the **EVALUATE** stage. Using the paper's exact values lets us verify Part II results against the published tables.

In [ ]:
Code(inspect.getsource(create_mock_portfolio), language="python")

In [ ]:
initiatives = create_mock_portfolio()

df = pd.DataFrame(initiatives)
df["gamma"] = df["confidence"].apply(calculate_gamma)
df

Initiative A has the highest confidence ($c = 0.90$, $\gamma = 0.10$) — its evidence is strong, so the penalty is minimal. Initiative D has the lowest confidence ($c = 0.40$, $\gamma = 0.60$) and will be excluded by the confidence threshold. Initiative E has large upside ($R_{\text{best}} = 18$) but zero downside ($R_{\text{worst}} = 0$) and medium confidence ($c = 0.50$).

## 2. Configuration

The allocation parameters are stored in `"config_allocation.yaml"`. The configuration specifies three constraints: the total budget, the minimum confidence threshold for eligibility, and the minimum worst-case portfolio return (the downside safeguard from Part I, equation 6).

In [ ]:
! cat config_allocation.yaml

In [ ]:
with open("config_allocation.yaml") as f:
    config = yaml.safe_load(f)

budget = config["budget"]
min_confidence = config["min_confidence"]
min_worst_return = config["min_worst_return"]

print(f"Budget:              {budget}")
print(f"Min confidence:      {min_confidence}")
print(f"Min worst return:    {min_worst_return}")

## 3. The confidence penalty

Before solving, we apply the confidence penalty from Part I, §3. The function `calculate_effective_returns()` computes $\gamma_i$ and the effective returns $\tilde{R}_{ij}$ for each initiative. We verify the results against the paper's Table 2.

In [ ]:
initiatives_with_returns = calculate_effective_returns(initiatives)

for init in initiatives_with_returns:
    eff = init["effective_returns"]
    print(
        f"  {init['id']}: gamma={init['gamma']:.2f}  "
        f"R_best={eff['best']:.1f}  R_med={eff['med']:.1f}  R_worst={eff['worst']:.1f}"
    )

In [ ]:
plot_confidence_penalty(initiatives_with_returns)

The penalty is largest for Initiative E ($\gamma = 0.50$): its best-case effective return drops from 18 to 9.0 because half the return is pulled toward the worst case of 0. Initiative A's returns barely change ($\gamma = 0.10$) — strong evidence preserves projected returns.

## 4. Preprocessing

The `preprocess()` function combines confidence filtering and effective return computation. Initiatives below the confidence threshold are excluded before the solver sees them.

In [ ]:
processed = preprocess(initiatives, min_confidence_threshold=min_confidence)

print(f"Initiatives before preprocessing: {len(initiatives)}")
print(f"Initiatives after preprocessing:  {len(processed)}")
print(f"Excluded: {[init['id'] for init in initiatives if init['id'] not in [p['id'] for p in processed]]}")

In [ ]:
plot_effective_returns_heatmap(processed)

Initiative D ($c = 0.40$) is excluded because it falls below the $c_{\min} = 0.50$ threshold. Four initiatives remain eligible: A, B, C, and E.

## 5. From theory to code

The `"config_allocation.yaml"` fields map directly to the theoretical constructs from Part I. The table below connects each configuration parameter to its role in the optimization formulation.

| Config Field | Part I Concept |
|---|---|
| `budget` | Total available resources $B$ (equation 5) |
| `min_confidence` | Minimum confidence threshold $c_{\min}$ (equation 3) |
| `min_worst_return` | Downside safeguard $R_{\min}^{\text{portfolio}}$ (equation 6) |

The initiative data fields map to the mathematical objects:

| Initiative Field | Part I Concept |
|---|---|
| `id` | Initiative index $i \in I$ |
| `cost` | Cost $b_i$ |
| `R_best`, `R_med`, `R_worst` | Baseline returns $R_{ij}$ under each scenario $s_j \in S$ |
| `confidence` | Confidence score $c_i$ from the **EVALUATE** stage |
| `gamma` | Penalty factor $\gamma_i = 1 - c_i$ |
| `effective_returns` | Confidence-adjusted returns $\tilde{R}_{ij}$ |

## 6. Minimax regret

We solve the minimax regret problem using the convenience wrapper `solve_minimax_regret()`. It preprocesses the initiatives (filtering and penalty computation) and solves the binary integer program from Part I, §4.

In [ ]:
minimax_result = solve_minimax_regret(
    initiatives,
    total_budget=budget,
    min_confidence_threshold=min_confidence,
    min_portfolio_worst_return=min_worst_return,
)

display_solver_result(minimax_result, "Minimax regret")

The solver selects initiatives A, C, and E at a total cost of 10 (exactly exhausting the budget). The maximum regret $\theta^* = 1.0$ means that in no scenario does this portfolio fall more than 1.0 below what was achievable. Initiative B is excluded despite being eligible — its inclusion would exceed the budget.

In [ ]:
plot_scenario_returns(minimax_result, "Minimax Regret: Portfolio Returns by Scenario")

The diamond markers show the optimal benchmark $V_j^*$ for each scenario — the best return achievable under budget constraints if we could pick the portfolio knowing which scenario would materialize. The gap between the bar and the diamond is the regret. Minimax regret minimizes the largest such gap across all scenarios.

## 7. Bayesian solver

The Bayesian decision rule from Part I, §5 maximizes weighted expected return. Different weight profiles express different beliefs about which scenario is most likely. We solve with three profiles: optimistic, balanced, and pessimistic.

In [ ]:
weight_profiles = {
    "Optimistic": {"best": 0.50, "med": 0.30, "worst": 0.20},
    "Balanced": {"best": 0.33, "med": 0.34, "worst": 0.33},
    "Pessimistic": {"best": 0.20, "med": 0.30, "worst": 0.50},
}

bayesian_results = {}
for name, weights in weight_profiles.items():
    solver = BayesianSolver(weights)
    result = solver(processed, total_budget=budget, min_portfolio_worst_return=min_worst_return)
    bayesian_results[name] = result
    display_solver_result(result, f"Bayesian ({name.lower()})")
    print()

## 8. Comparing decision rules

We compare the minimax regret solution with the three Bayesian profiles. The comparison reveals how the portfolio choice depends on the decision-maker's attitude toward uncertainty.

In [ ]:
all_results = [("Minimax regret", minimax_result)]
for name, result in bayesian_results.items():
    all_results.append((f"Bayesian ({name.lower()})", result))

plot_portfolio_comparison(all_results)

In [ ]:
print("Selected initiatives by decision rule")
print("=" * 50)
for name, result in all_results:
    selected = result["selected_initiatives"]
    cost = result["total_cost"]
    print(f"  {name + ':':30s} {selected}  (cost={cost:.0f})")

## 9. How confidence shapes allocation

The confidence penalty creates a direct link between evidence quality and resource allocation. To see this incentive effect, we vary Initiative E's confidence from 0.50 to 0.95 and observe how the minimax regret solution changes. As evidence quality improves, the effective returns rise and the portfolio becomes more attractive.

In [ ]:
confidence_values = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

print("Effect of Initiative E's confidence on minimax regret solution")
print("=" * 65)
print(f"  {'c_E':>5s}  {'gamma':>5s}  {'Selected':>15s}  {'theta*':>7s}  {'Worst return':>13s}")
print("-" * 65)

for c_e in confidence_values:
    modified = copy.deepcopy(initiatives)
    for init in modified:
        if init["id"] == "E":
            init["confidence"] = c_e

    result = solve_minimax_regret(
        modified,
        total_budget=budget,
        min_confidence_threshold=min_confidence,
        min_portfolio_worst_return=min_worst_return,
    )

    selected = ", ".join(result["selected_initiatives"])
    theta = result["objective_value"]
    worst = result["total_actual_returns"].get("worst", 0.0)
    gamma_e = calculate_gamma(c_e)
    theta_str = f"{theta:.2f}" if theta is not None else "N/A"
    print(f"  {c_e:5.2f}  {gamma_e:5.2f}  {selected:>15s}  {theta_str:>7s}  {worst:13.1f}")

This table demonstrates the incentive effect: as the confidence in Initiative E's evidence improves, its effective returns increase and the maximum regret $\theta^*$ may decrease. Better measurement does not just improve estimates — it changes which initiatives get funded and how much regret the portfolio faces.

## Additional resources

- **Eisenhauer, P. (2025)**. Science-backed decisions for high-stakes investments under uncertainty.

- **eisenhauer.io (2026)**. [impact-engine-allocate documentation](https://eisenhauerio.github.io/tools-impact-engine-allocate/). Usage, configuration, and system design.